# 05 — Centroidal力学

## 目的
NMPCが予測する「胴体と関節の縮約力学」を、接触力の合力・合momentから理解する。
本実装の既定は単一剛体だけではなく `FullCentroidalDynamics` であり、関節運動の影響を残す。

実装対応:
- `legged_interface/src/LeggedInterface.cpp`
- OCS2 `LeggedRobotDynamicsAD`（外部。完全な成分ODEはこのworkspaceでは未照合）


## 検証範囲に関する必須注記

このprojectでは **ROS2 portを作成・compile・実行していない**。したがってROS2 parityは
**NOT VERIFIED / FAIL-CLOSED** である。上流commit `a7f381c0367e98e31c01336e678eef47e304d40d` はROS1実装であり、
project所有MuJoCo adapterはOCS2のhorizon SQPを瞬時force plannerへ、
Pinocchio/qpOASES WBCをMuJoCo acceleration inverse dynamicsへ置換し、
元のestimator/hardware経路も持たない。保存済み30 scenario dataが示すのはadapter挙動だけで、
上流 `legged_control` の性能でもROS2移行の検証でもない。


In [1]:
# 背景: 接触力の合力・合momentがcentroidal運動を決める。目的: 力と運動量変化の物理整合を検査するため、`from pathlib import Path` の依存を明示して再現可能な実行環境を作る。
from pathlib import Path
# 背景: 接触力の合力・合momentがcentroidal運動を決める。目的: 力と運動量変化の物理整合を検査するため、`import numpy as np` の依存を明示して再現可能な実行環境を作る。
import numpy as np
# 背景: 接触力の合力・合momentがcentroidal運動を決める。目的: 力と運動量変化の物理整合を検査するため、`import matplotlib.pyplot as plt` の依存を明示して再現可能な実行環境を作る。
import matplotlib.pyplot as plt

# 背景: 接触力の合力・合momentがcentroidal運動を決める。目的: 力と運動量変化の物理整合を検査するため、`ROOT` を後続計算で使う明示的な中間量として設定する。 数式: `ROOT = Path.cwd()` の演算・変換をPythonで評価する。
ROOT = Path.cwd()
# 背景: 接触力の合力・合momentがcentroidal運動を決める。目的: 力と運動量変化の物理整合を検査するため、`for candidate in [ROOT, *ROOT.parents]:` の反復範囲を固定して各sampleを処理する。 数式: `for candidate in [ROOT, *ROOT.parents]:` の演算・変換をPythonで評価する。
for candidate in [ROOT, *ROOT.parents]:
    # 背景: 接触力の合力・合momentがcentroidal運動を決める。目的: 力と運動量変化の物理整合を検査するため、`if (candidate / "pyproject.toml").exists():` の条件で安全側の実行分岐を選ぶ。 数式: `if (candidate / "pyproject.toml").exists():` の演算・変換をPythonで評価する。
    if (candidate / "pyproject.toml").exists():
        # 背景: 接触力の合力・合momentがcentroidal運動を決める。目的: 力と運動量変化の物理整合を検査するため、`ROOT` を後続計算で使う明示的な中間量として設定する。 数式: `ROOT = candidate` の演算・変換をPythonで評価する。
        ROOT = candidate
        # 背景: 接触力の合力・合momentがcentroidal運動を決める。目的: 力と運動量変化の物理整合を検査するため、`break` をこの章の処理順に沿って実行する。
        break

# 背景: 接触力の合力・合momentがcentroidal運動を決める。目的: 力と運動量変化の物理整合を検査するため、`np.set_printoptions(precision` を後続計算で使う明示的な中間量として設定する。 数式: `np.set_printoptions(precision=4, suppress=True)` の演算・変換をPythonで評価する。
np.set_printoptions(precision=4, suppress=True)
# 背景: 接触力の合力・合momentがcentroidal運動を決める。目的: 力と運動量変化の物理整合を検査するため、直前の式・構造へ `plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True})` の要素または終端を対応付ける。
plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True})
# 背景: 接触力の合力・合momentがcentroidal運動を決める。目的: 力と運動量変化の物理整合を検査するため、`print("repository:", ROOT)` の観測値を表示して判定根拠を残す。
print("repository:", ROOT)


repository: /home/takuya/work/mpc_dog


接触力 \(f_i\) とCoMから足へのlever arm \(r_i\) に対して
\[
m\dot v=\sum_i f_i+m g,\qquad
\dot L=\sum_i r_i\times f_i.
\]
静止なら \(\sum f_{i,z}=mg\)。4脚等分は初期guessにはなるが、加速・姿勢moment・接触数が
変われば一般には等分でない。


In [2]:
# 背景: 接触力の合力・合momentがcentroidal運動を決める。目的: 力と運動量変化の物理整合を検査するため、`mass` を後続計算で使う明示的な中間量として設定する。 数式: `mass = 12.5` の演算・変換をPythonで評価する。
mass = 12.5
# 背景: 接触力の合力・合momentがcentroidal運動を決める。目的: 力と運動量変化の物理整合を検査するため、`g` を後続計算で使う明示的な中間量として設定する。 数式: `g = np.array([0.0, 0.0, -9.81])` の演算・変換をPythonで評価する。
g = np.array([0.0, 0.0, -9.81])
# 背景: 接触力の合力・合momentがcentroidal運動を決める。目的: 力と運動量変化の物理整合を検査するため、`feet` を後続計算で使う明示的な中間量として設定する。 数式: `feet = np.array([` の演算・変換をPythonで評価する。
feet = np.array([
    # 背景: 接触力の合力・合momentがcentroidal運動を決める。目的: 力と運動量変化の物理整合を検査するため、`[ 0.25, 0.15, -0.30], # LF` をこの章の処理順に沿って実行する。 数式: `[ 0.25, 0.15, -0.30], # LF` の演算・変換をPythonで評価する。
    [ 0.25,  0.15, -0.30],  # LF
    # 背景: 接触力の合力・合momentがcentroidal運動を決める。目的: 力と運動量変化の物理整合を検査するため、`[ 0.25, -0.15, -0.30], # RF` をこの章の処理順に沿って実行する。 数式: `[ 0.25, -0.15, -0.30], # RF` の演算・変換をPythonで評価する。
    [ 0.25, -0.15, -0.30],  # RF
    # 背景: 接触力の合力・合momentがcentroidal運動を決める。目的: 力と運動量変化の物理整合を検査するため、`[-0.25, 0.15, -0.30], # LH` をこの章の処理順に沿って実行する。 数式: `[-0.25, 0.15, -0.30], # LH` の演算・変換をPythonで評価する。
    [-0.25,  0.15, -0.30],  # LH
    # 背景: 接触力の合力・合momentがcentroidal運動を決める。目的: 力と運動量変化の物理整合を検査するため、`[-0.25, -0.15, -0.30], # RH` をこの章の処理順に沿って実行する。 数式: `[-0.25, -0.15, -0.30], # RH` の演算・変換をPythonで評価する。
    [-0.25, -0.15, -0.30],  # RH
# 背景: 接触力の合力・合momentがcentroidal運動を決める。目的: 力と運動量変化の物理整合を検査するため、直前の式・構造へ `])` の要素または終端を対応付ける。
])
# 背景: 接触力の合力・合momentがcentroidal運動を決める。目的: 力と運動量変化の物理整合を検査するため、`forces` を後続計算で使う明示的な中間量として設定する。 数式: `forces = np.tile(np.array([0, 0, mass*9.81/4]), (4, 1))` の演算・変換をPythonで評価する。
forces = np.tile(np.array([0, 0, mass*9.81/4]), (4, 1))

# 背景: 接触力の合力・合momentがcentroidal運動を決める。目的: 力と運動量変化の物理整合を検査するため、`net_force` を後続計算で使う明示的な中間量として設定する。 数式: `net_force = forces.sum(axis=0) + mass*g` の演算・変換をPythonで評価する。
net_force = forces.sum(axis=0) + mass*g
# 背景: 接触力の合力・合momentがcentroidal運動を決める。目的: 力と運動量変化の物理整合を検査するため、`net_moment` を後続計算で使う明示的な中間量として設定する。 数式: `net_moment = np.cross(feet, forces).sum(axis=0)` の演算・変換をPythonで評価する。
net_moment = np.cross(feet, forces).sum(axis=0)
# 背景: 接触力の合力・合momentがcentroidal運動を決める。目的: 力と運動量変化の物理整合を検査するため、`print("net force incl. gravity [N]:", net_force)` の観測値を表示して判定根拠を残す。
print("net force incl. gravity [N]:", net_force)
# 背景: 接触力の合力・合momentがcentroidal運動を決める。目的: 力と運動量変化の物理整合を検査するため、`print("net moment about CoM [N m]:", net_moment)` の観測値を表示して判定根拠を残す。
print("net moment about CoM [N m]:", net_moment)
# 背景: 接触力の合力・合momentがcentroidal運動を決める。目的: 力と運動量変化の物理整合を検査するため、`assert np.allclose(net_force, 0)` を不変条件として即時検査する。 数式: `assert np.allclose(net_force, 0)` の演算・変換をPythonで評価する。
assert np.allclose(net_force, 0)
# 背景: 接触力の合力・合momentがcentroidal運動を決める。目的: 力と運動量変化の物理整合を検査するため、`assert np.allclose(net_moment, 0)` を不変条件として即時検査する。 数式: `assert np.allclose(net_moment, 0)` の演算・変換をPythonで評価する。
assert np.allclose(net_moment, 0)


net force incl. gravity [N]: [0. 0. 0.]
net moment about CoM [N m]: [0. 0. 0.]


In [3]:
# 対角2脚支持で前進加速度0.5 m/s^2を作る例。
# 背景: 接触力の合力・合momentがcentroidal運動を決める。目的: 力と運動量変化の物理整合を検査するため、`contact` を後続計算で使う明示的な中間量として設定する。 数式: `contact = np.array([1, 0, 0, 1], dtype=bool)` の演算・変換をPythonで評価する。
contact = np.array([1, 0, 0, 1], dtype=bool)
# 背景: 接触力の合力・合momentがcentroidal運動を決める。目的: 力と運動量変化の物理整合を検査するため、`forces_trot` を後続計算で使う明示的な中間量として設定する。 数式: `forces_trot = np.zeros((4, 3))` の演算・変換をPythonで評価する。
forces_trot = np.zeros((4, 3))
# 背景: 接触力の合力・合momentがcentroidal運動を決める。目的: 力と運動量変化の物理整合を検査するため、`forces_trot[contact, 0]` を後続計算で使う明示的な中間量として設定する。 数式: `forces_trot[contact, 0] = mass * 0.5 / contact.sum()` の演算・変換をPythonで評価する。
forces_trot[contact, 0] = mass * 0.5 / contact.sum()
# 背景: 接触力の合力・合momentがcentroidal運動を決める。目的: 力と運動量変化の物理整合を検査するため、`forces_trot[contact, 2]` を後続計算で使う明示的な中間量として設定する。 数式: `forces_trot[contact, 2] = mass * 9.81 / contact.sum()` の演算・変換をPythonで評価する。
forces_trot[contact, 2] = mass * 9.81 / contact.sum()
# 背景: 接触力の合力・合momentがcentroidal運動を決める。目的: 力と運動量変化の物理整合を検査するため、`a_com` を後続計算で使う明示的な中間量として設定する。 数式: `a_com = forces_trot.sum(axis=0) / mass + g` の演算・変換をPythonで評価する。
a_com = forces_trot.sum(axis=0) / mass + g
# 背景: 接触力の合力・合momentがcentroidal運動を決める。目的: 力と運動量変化の物理整合を検査するため、`moment` を後続計算で使う明示的な中間量として設定する。 数式: `moment = np.cross(feet, forces_trot).sum(axis=0)` の演算・変換をPythonで評価する。
moment = np.cross(feet, forces_trot).sum(axis=0)
# 背景: 接触力の合力・合momentがcentroidal運動を決める。目的: 力と運動量変化の物理整合を検査するため、`print("forces [N]:\n", forces_trot)` の観測値を表示して判定根拠を残す。
print("forces [N]:\n", forces_trot)
# 背景: 接触力の合力・合momentがcentroidal運動を決める。目的: 力と運動量変化の物理整合を検査するため、`print("CoM acceleration [m/s^2]:", a_com)` の観測値を表示して判定根拠を残す。 数式: `print("CoM acceleration [m/s^2]:", a_com)` の演算・変換をPythonで評価する。
print("CoM acceleration [m/s^2]:", a_com)
# 背景: 接触力の合力・合momentがcentroidal運動を決める。目的: 力と運動量変化の物理整合を検査するため、`print("moment [N m]:", moment)` の観測値を表示して判定根拠を残す。
print("moment [N m]:", moment)


forces [N]:
 [[ 3.125   0.     61.3125]
 [ 0.      0.      0.    ]
 [ 0.      0.      0.    ]
 [ 3.125   0.     61.3125]]
CoM acceleration [m/s^2]: [0.5 0.  0. ]
moment [N m]: [ 0.    -1.875  0.   ]


## Full centroidal と SRBD
`centroidalModelType=0` はfull centroidal。状態は正規化centroidal momentumと
base pose、joint anglesで、入力後半のjoint velocityを通して形状変化も予測へ入る。
SRBDへ切替える設定値はあるが、A1既定ではない。

### 数式変更前の検査
1. 静止4脚で重力が相殺される。
2. 左右対称力でroll momentが0。
3. 接触していない脚の力が0。
4. 単位をN、N m、kg、m、sへ統一。


## 章固有の背景
                NMPCは18自由度の全運動をそのまま積分せず、全身の運動量と形状へ縮約して未来を予測する。

                ## 章固有の目的
                接触力からlinear/angular momentum rateが生じる式と、A1既定full-centroidal選択を確認する。

                ## この章のASCIIデータフロー
                ```text
                x=[h/m,base pose,q], u=[four forces,dq]
        -> LeggedRobotDynamicsAD / centroidal model
        -> xdot=[sum(F)/m+g, sum(r x F)/m, pose rate, dq]
                ```

                ## 上流C++ / faithful pseudocode と数式の行対応
                ```cpp
                // external/legged_control/legged_interface/src/LeggedInterface.cpp
centroidalModelInfo = createCentroidalModelInfo(...); // task.info modelType=0
dynamicsPtr.reset(new LeggedRobotDynamicsAD(...));     // xdot=f(x,u)
// faithful equation map (OCS2内部成分実装はこのworkspaceで未照合)
hDot_linear = sum_i(f_i) + m*g;       // m*vdot = Σf + mg
hDot_angular = sum_i(r_i.cross(f_i)); // Ldot = Σ(r_i×f_i)
qDot_joint = u.tail(12);              // 入力後半は関節速度
                ```

                **事実のラベル**: `external/legged_control/` の記述はcommit
                `a7f381c0367e98e31c01336e678eef47e304d40d` の上流実装事実。数式展開はそのinterfaceを説明する理論。
                `src/legged_control_mujoco/` に言及した行はproject所有adapterの実装であり、
                ROS1/OCS2 SQP原実装とは同一ではない。

                ## 章固有の結論
                合力が並進、CoMまわりの合momentが角運動量を変える。A1既定はSRBDではなくfull centroidalである。
